# Practice Lab: GPT and Decoder Models (Lesson 4.4)

Module 4 · 8 exercises · ~120-150 min
**Needs:** transformers, google-genai


# Lesson 4.4: GPT and Decoder Models

8 exercises: 2E + 3M + 3H
**Needs:** transformers, google-genai, Gemini API key


In [ ]:
!pip install transformers google-genai -q
import numpy as np


---
## Exercise 6 (Hard): Custom Sampler
Implement temperature + top-k + top-p + rep penalty.


In [ ]:
# YOUR CODE
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

# Mock logits for 10 words
words = ["mat","floor","roof","moon","pizza",
         "cat","the","dog","sky","hat"]
logits = np.array([5.0, 4.2, 2.5, 0.8, -1.0,
                   3.1, 4.5, 1.2, 0.3, 2.8])

# TODO: implement full sampling pipeline
# TODO: show distribution after each step


<details><summary>Solution</summary>


In [ ]:
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

words = ["mat","floor","roof","moon","pizza",
         "cat","the","dog","sky","hat"]
logits = np.array([5.0, 4.2, 2.5, 0.8, -1.0,
                   3.1, 4.5, 1.2, 0.3, 2.8])

def custom_sample(logits, temp=0.7, top_k=5,
                  top_p=0.9, rep_penalty=1.2,
                  generated=None):
    # Step 1: Temperature
    scaled = logits / temp
    p1 = softmax(scaled)
    print(f"  After temp={temp}:")
    for w, p in zip(words, p1):
        if p > 0.01:
            print(f"    {w:8s} {p:.1%}")

    # Step 2: Top-k
    if top_k > 0:
        thresh = np.sort(scaled)[-top_k]
        scaled[scaled < thresh] = -1e9
    p2 = softmax(scaled)
    surviving = sum(p2 > 0.001)
    print(f"  After top-k={top_k}: "
          f"{surviving} tokens survive")

    # Step 3: Top-p
    probs = p2.copy()
    if top_p < 1.0:
        sidx = np.argsort(probs)[::-1]
        csum = np.cumsum(probs[sidx])
        cutoff = np.searchsorted(csum, top_p) + 1
        keep = sidx[:cutoff]
        mask = np.zeros_like(probs)
        mask[keep] = probs[keep]
        probs = mask / mask.sum()
    surviving = sum(probs > 0.001)
    print(f"  After top-p={top_p}: "
          f"{surviving} tokens survive")

    # Step 4: Repetition penalty
    if generated and rep_penalty != 1.0:
        for gid in set(generated):
            if gid < len(probs):
                probs[gid] /= rep_penalty
        probs /= probs.sum()
    
    return np.random.choice(len(probs), p=probs)

print("Full sampling pipeline:\n")
chosen = custom_sample(
    logits.copy(), temp=0.7, top_k=5,
    top_p=0.9, rep_penalty=1.2, generated=[0, 6])
print(f"\nChosen: '{words[chosen]}'")

# Sample 100 times
print("\n100 samples distribution:")
counts = {}
for _ in range(100):
    idx = custom_sample(
        logits.copy(), temp=0.7, top_k=5,
        top_p=0.9, rep_penalty=1.0, generated=[])
    w = words[idx]
    counts[w] = counts.get(w, 0) + 1
for w, c in sorted(counts.items(),
                   key=lambda x: -x[1]):
    print(f"  {w:8s}: {c}%")


</details>


---
## Exercise 1 (Easy): Temperature Experiment
Generate a fun fact about India at temperatures 0.1, 0.5, 1.0, 1.5 (3 runs each). Rate coherence and find where output becomes nonsensical. **Needs GEMINI_API_KEY.**

In [ ]:
# YOUR CODE
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

prompt = "Tell me a fun fact about India in one sentence."

for temp in [0.1, 0.5, 1.0, 1.5]:
    print(f"\n  Temperature = {temp}")
    for run in range(3):
        resp = client.models.generate_content(
            model="gemini-3-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=temp,
                max_output_tokens=60))
        print(f"    Run {run+1}: {resp.text.strip()[:80]}")

# TODO: rate coherence 1-5 for each temperature
# TODO: measure variation between runs

---
## Exercise 2 (Easy): Logprobs Explorer
Use GPT-2 to show top-5 next-token probabilities for 5 prompts. Compare factual vs creative confidence, then compute perplexity for one sentence. **Keyless (GPT-2).**

In [ ]:
# YOUR CODE
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model.eval()

prompts = [
    "The president of India is",
    "def fibonacci(",
    "2 + 2 =",
    "Once upon a time in a",
    "The meaning of life is",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)
    top5 = torch.topk(probs, 5)
    print(f"\n  '{prompt}'")
    for i in range(5):
        tok = tokenizer.decode(top5.indices[i])
        p = top5.values[i].item()
        lp = torch.log(top5.values[i]).item()
        print(f"    {i+1}. '{tok}' "
              f"prob={p:.1%} logprob={lp:.2f}")

# TODO: compare confidence across prompts
# TODO: compute perplexity for one sentence

---
## Exercise 3 (Medium): In-Context Learning Explorer
Test GPT-2 sentiment classification at 0/1/3/5-shot. Track accuracy vs number of examples and find the plateau. **Keyless (GPT-2).**

In [ ]:
# YOUR CODE
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2",
                     max_new_tokens=5)

examples = [
    ("Great movie, loved it!", "positive"),
    ("Terrible, waste of time.", "negative"),
    ("Amazing acting and plot.", "positive"),
    ("Boring and predictable.", "negative"),
    ("Masterpiece of cinema.", "positive"),
]

test_reviews = [
    ("Best film of the year!", "positive"),
    ("I fell asleep.", "negative"),
    ("Incredible performances.", "positive"),
    ("Don't waste your money.", "negative"),
    ("A joy to watch.", "positive"),
]

# TODO: for 0, 1, 3, 5 shot: build prompt, test accuracy
# TODO: plot accuracy vs shot count

---
## Exercise 4 (Medium): Structured Output Extractor
Extract {name, age, city} from text with Gemini `response_schema`, then compare against free-form JSON extraction. **Needs GEMINI_API_KEY.**

In [ ]:
# YOUR CODE
from google import genai
from google.genai import types
import json, os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
        "city": {"type": "string"},
    }
}

texts = [
    "Priya Sharma, 28, lives in Mumbai.",
    "Rajesh is a 45 year old from Chennai.",
    "Anita (32) recently moved to Bangalore.",
    "Meet Vikram, age 55, Hyderabad resident.",
    "Deepa, twenty-three, Delhi.",
]

# TODO: extract with schema (structured)
# TODO: extract without schema (free-form)
# TODO: compare valid JSON rate

---
## Exercise 5 (Medium): Streaming Story Generator
Generate a short story with `generate_content_stream`, measuring time-to-first-token and total time vs non-streaming. **Needs GEMINI_API_KEY.**

In [ ]:
# YOUR CODE
from google import genai
import time, os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

genre = "mystery"  # or: sci-fi, romance, comedy
prompt = f"Write a 100-word {genre} story."

# Streaming
t0 = time.time()
first_token_time = None
for chunk in client.models.generate_content_stream(
        model="gemini-3-flash", contents=prompt):
    if first_token_time is None:
        first_token_time = time.time() - t0
    print(chunk.text, end="", flush=True)
total_stream = time.time() - t0

print(f"\n\nTime-to-first-token: {first_token_time:.2f}s")
print(f"Total streaming:     {total_stream:.2f}s")

# TODO: compare with non-streaming
# TODO: test multiple genres

---
## Exercise 7 (Hard): Perplexity Calculator
Compute GPT-2 perplexity across domains (Wikipedia / Reddit / Python) and explain what the ranking reveals about training data. **Keyless (GPT-2).**

In [ ]:
# YOUR CODE
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model.eval()

texts = {
    "Wikipedia": (
        "The Taj Mahal is an ivory-white marble "
        "mausoleum on the south bank of the Yamuna "
        "river in Agra, India."),
    "Reddit": (
        "lol this is so true, literally me every "
        "monday morning trying to adult"),
    "Python": (
        "def quicksort(arr):\n"
        "    if len(arr) <= 1: return arr\n"
        "    pivot = arr[0]\n"
        "    return quicksort([x for x in arr[1:]"
        " if x < pivot]) + [pivot]"),
}

for name, text in texts.items():
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs, labels=inputs["input_ids"])
    ppl = torch.exp(out.loss).item()
    print(f"  {name:12s}: perplexity = {ppl:.1f}")

# TODO: explain which domain fits GPT-2 best
# TODO: bonus: test on Hindi text

---
## Exercise 8 (Challenge): Model Comparison Dashboard
Compare Gemini Flash vs local GPT-2 on speed, quality, and cost. Print a comparison table. **Gemini path needs GEMINI_API_KEY.**

In [ ]:
# YOUR CODE
import time

prompt = "Explain quantum computing in 3 sentences."

# Model 1: Gemini Flash (API)
# Model 2: GPT-2 (local, HuggingFace)
# Model 3: (optional) Ollama local model

results = []
# TODO: generate with each model, time it
# TODO: count tokens, compute tokens/sec
# TODO: estimate cost per 1M tokens
# TODO: print comparison table